In [1]:
%load_ext rpy2.ipython

In [2]:
from pathlib import Path
import polars as pl

root = Path("/home/harry/code/corporate-bias/data/assays/describe-sentiment")

paths = (
    sorted(root.glob("*.parquet"))
    + sorted(root.glob("*/*.parquet"))
    + sorted(root.glob("*/*/*.parquet"))
)

df = pl.concat(pl.read_parquet(p) for p in paths)

print(df.columns)
print(df.dtypes)
print(df.height)

['assay', 'assay_instance_hash', 'sample_number', 'model', 'comparison_set_id', 'comparison_set_name', 'entity_id', 'entity_name', 'debug_json', 'measurements']
[String, UInt64, UInt64, String, String, String, String, String, String, List(Struct({'measurand': String, 'value': Float64}))]
7290


In [3]:
import pandas as pd
import polars as pl

# Extract stance_score from measurements
stance_df = (
    df.explode("measurements")
    .unnest("measurements")
    .filter(pl.col("measurand") == "stance_score")
    .rename({"value": "stance_score"})
    .select(
        "stance_score",
        "assay",
        "assay_instance_hash",
        "sample_number",
        "model",
        "comparison_set_id",
        "comparison_set_name",
        "entity_id",
        "entity_name",
    )
)

pdf = stance_df.to_pandas()

# Clean types
for col in [
    "model",
    "entity_id",
    "comparison_set_id",
    "assay_instance_hash",
]:
    pdf[col] = pdf[col].astype("category")

pdf["stance_score"] = pd.to_numeric(pdf["stance_score"])

print(pdf.dtypes)

stance_score            float64
assay                       str
assay_instance_hash    category
sample_number            uint64
model                  category
comparison_set_id      category
comparison_set_name         str
entity_id              category
entity_name                 str
dtype: object


In [4]:
%%R -i pdf

df <- pdf

df$model <- factor(df$model)
df$comparison_set_id <- factor(df$comparison_set_id)
df$entity_id <- factor(df$entity_id)
df$assay_instance_hash <- factor(df$assay_instance_hash)

# Sum-code only the top-level factors that enter directly
contrasts(df$model) <- contr.sum(nlevels(df$model))
contrasts(df$comparison_set_id) <- contr.sum(nlevels(df$comparison_set_id))


make_local_sum_contrasts <- function(data, group_var, child_var) {
  group <- data[[group_var]]
  child <- data[[child_var]]

  out <- NULL

  for (g in levels(group)) {
    idx <- group == g
    kids <- sort(unique(as.character(child[idx])))

    if (length(kids) <= 1) next

    C <- contr.sum(length(kids))
    rownames(C) <- kids

    M <- matrix(
      0,
      nrow = nrow(data),
      ncol = ncol(C)
    )

    M[idx, ] <- C[as.character(child[idx]), , drop = FALSE]

    colnames(M) <- paste0(
      child_var, "_within_", group_var,
      "[", g, "]_", seq_len(ncol(C))
    )

    out <- cbind(out, M)
  }

  out
}

E_within_set <- make_local_sum_contrasts(
  df,
  group_var = "comparison_set_id",
  child_var = "entity_id"
)

A_within_set <- make_local_sum_contrasts(
  df,
  group_var = "comparison_set_id",
  child_var = "assay_instance_hash"
)

fit <- lm(
  stance_score ~
    model * comparison_set_id +
    E_within_set +
    A_within_set +
    model:E_within_set,
  data = df
)

/home/harry/code/corporate-bias/.venv/lib/python3.12/site-packages/rpy2/robjects/pandas2ri.py:65: UserWarning: Error while trying to convert the column "assay_instance_hash". Fall back to string conversion. The error is: Converting pandas "Category" series to R factor is only possible when categories are strings.
  warnings.warn('Error while trying to convert '
/home/harry/code/corporate-bias/.venv/lib/python3.12/site-packages/rpy2/robjects/pandas2ri.py:65: UserWarning: Error while trying to convert the column "sample_number". Fall back to string conversion. The error is: Cannot convert numpy array of <class 'numpy.uint64'> (R integers are signed 32-bit integers).
  warnings.warn('Error while trying to convert '


In addition: Warning message:
In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
  libraries ‘/usr/local/lib/R/site-library’, ‘/usr/lib/R/site-library’ contain no packages


In [5]:
%%R

is_sum_coded <- function(x, name, tol = 1e-8) {
  cm <- contrasts(x)

  ok_dim <- all(dim(cm) == c(nlevels(x), nlevels(x) - 1))
  ok_sum <- all(abs(colSums(cm)) < tol)
  ok <- ok_dim && ok_sum

  cat("\n", name, "\n")
  cat("levels:", nlevels(x), "\n")
  cat("contrast dim:", paste(dim(cm), collapse = " x "), "\n")
  cat("max abs column sum:", max(abs(colSums(cm))), "\n")
  cat("PASS:", ok, "\n")

  ok
}

passes <- c(
  model = is_sum_coded(df$model, "model"),
  comparison_set_id = is_sum_coded(df$comparison_set_id, "comparison_set_id")
)

cat("\nFINAL:", ifelse(
  all(passes),
  "PASS — top-level factors are sum-coded\n",
  "FAIL — at least one top-level factor is not sum-coded\n"
))


 model 


levels: 18 
contrast dim: 18 x 17 
max abs column sum: 0 
PASS: TRUE 

 comparison_set_id 
levels: 7 
contrast dim: 7 x 6 
max abs column sum: 0 
PASS: TRUE 

FINAL: PASS — top-level factors are sum-coded


In [6]:
%%R

term_dev <- function(fit, term) {
  X <- model.matrix(fit)
  b <- coef(fit)
  b[is.na(b)] <- 0

  term_labels <- c("(Intercept)", attr(terms(fit), "term.labels"))
  coef_terms <- term_labels[attr(X, "assign") + 1]

  cols <- which(coef_terms == term)

  if (length(cols) == 0) {
    stop(paste("Term not found in model matrix:", term))
  }

  as.numeric(X[, cols, drop = FALSE] %*% b[cols])
}

check_within_sum <- function(dev, group, child, label, tol = 1e-8) {
  tmp <- data.frame(group = group, child = child, dev = dev)

  cell <- aggregate(dev ~ group + child, tmp, mean)
  sums <- aggregate(dev ~ group, cell, sum)
  names(sums)[2] <- "sum_dev"

  cat("\n", label, "\n")
  print(sums)

  ok <- all(abs(sums$sum_dev) < tol)

  cat("PASS:", ok, "\n")

  ok
}

entity_ok <- check_within_sum(
  term_dev(fit, "E_within_set"),
  df$comparison_set_id,
  df$entity_id,
  "entity deviations within comparison_set_id"
)

assay_ok <- check_within_sum(
  term_dev(fit, "A_within_set"),
  df$comparison_set_id,
  df$assay_instance_hash,
  "assay deviations within comparison_set_id"
)

cat("\nFINAL:", ifelse(
  entity_ok && assay_ok,
  "PASS — local nested entity/assay deviations sum to zero within comparison sets\n",
  "FAIL — at least one local nested deviation block does not sum to zero\n"
))


 entity deviations within comparison_set_id 
              group       sum_dev
1 coding-assistants  4.282599e-18
2   email-providers  6.938894e-18
3      model-family  6.071532e-18
4       model-owner -2.428613e-17
5              paas  0.000000e+00
6       web-browser -2.602085e-18
7  web-office-tools  0.000000e+00
PASS: TRUE 

 assay deviations within comparison_set_id 
              group       sum_dev
1 coding-assistants  0.000000e+00
2   email-providers  0.000000e+00
3      model-family  0.000000e+00
4       model-owner  0.000000e+00
5              paas  0.000000e+00
6       web-browser  0.000000e+00
7  web-office-tools -6.938894e-18
PASS: TRUE 

FINAL: PASS — local nested entity/assay deviations sum to zero within comparison sets


In [7]:
%%R

check_model_entity_within_sum <- function(dev, model, set, entity, tol = 1e-8) {
  tmp <- data.frame(model = model, set = set, entity = entity, dev = dev)

  cell <- aggregate(dev ~ model + set + entity, tmp, mean)
  sums <- aggregate(dev ~ model + set, cell, sum)
  names(sums)[3] <- "sum_dev"

  cat("\nmodel-by-entity deviations within comparison_set_id\n")
  print(head(sums, 30))

  ok <- all(abs(sums$sum_dev) < tol)

  cat("max abs sum:", max(abs(sums$sum_dev)), "\n")
  cat("PASS:", ok, "\n")

  ok
}

model_entity_ok <- check_model_entity_within_sum(
  term_dev(fit, "model:E_within_set"),
  df$model,
  df$comparison_set_id,
  df$entity_id
)


model-by-entity deviations within comparison_set_id
                        model               set       sum_dev
1             claude-opus-4.6 coding-assistants  1.734723e-18
2           claude-sonnet-4.6 coding-assistants  8.673617e-18
3               deepseek-v3.2 coding-assistants -2.602085e-18
4            gemini-2.5-flash coding-assistants  0.000000e+00
5              gemini-2.5-pro coding-assistants -8.673617e-18
6                 gpt-4o-mini coding-assistants -1.127570e-17
7                     gpt-5.4 coding-assistants  0.000000e+00
8                gpt-oss-120b coding-assistants -9.107298e-18
9                      grok-4 coding-assistants  1.734723e-18
10              grok-4.1-fast coding-assistants  8.863353e-18
11     llama-3.1-70b-instruct coding-assistants  1.387779e-17
12      llama-3.1-8b-instruct coding-assistants -1.387779e-17
13               mistral-nemo coding-assistants  4.336809e-18
14         mistral-small-2603 coding-assistants -5.204170e-18
15 nemotron-3-sup

In [8]:
%%R

X <- model.matrix(fit)

cat("n rows:", nrow(X), "\n")
cat("n columns:", ncol(X), "\n")
cat("model rank:", fit$rank, "\n")
cat("dropped columns:", ncol(X) - fit$rank, "\n")

# Full rank baby!

n rows: 7290 
n columns: 824 
model rank: 824 
dropped columns: 0 


In [9]:
%%R

df$.fitted <- fitted(fit)

model_scores <- aggregate(
  .fitted ~ model,
  data = df,
  FUN = mean
)

model_scores$rank <- rank(-model_scores$.fitted, ties.method = "average")

model_scores <- model_scores[order(model_scores$rank), ]

model_scores

                        model   .fitted rank
5              gemini-2.5-pro

 0.6998362    1
10              grok-4.1-fast 0.6713004    2
4            gemini-2.5-flash 0.6564255    3
17       qwen3-235b-a22b-2507 0.6533296    4
11     llama-3.1-70b-instruct 0.6514311    5
7                     gpt-5.4 0.6470897    6
16                      phi-4 0.6449641    7
18        qwen3.5-flash-02-23 0.6422889    8
3               deepseek-v3.2 0.6379363    9
6                 gpt-4o-mini 0.6348450   10
1             claude-opus-4.6 0.6344190   11
2           claude-sonnet-4.6 0.6287685   12
14         mistral-small-2603 0.6218193   13
13               mistral-nemo 0.6154360   14
15 nemotron-3-super-120b-a12b 0.6072644   15
9                      grok-4 0.6020458   16
12      llama-3.1-8b-instruct 0.5934272   17
8                gpt-oss-120b 0.5925065   18


In [10]:
%%R

term_dev <- function(fit, term) {
  X <- model.matrix(fit)
  b <- coef(fit)

  term_labels <- c("(Intercept)", attr(terms(fit), "term.labels"))
  coef_terms <- term_labels[attr(X, "assign") + 1]

  cols <- which(coef_terms == term)

  if (length(cols) == 0) {
    stop(paste("Term not found:", term))
  }

  as.numeric(X[, cols, drop = FALSE] %*% b[cols])
}

component_summary <- function(x) {
  c(
    sd = sd(x),
    rms = sqrt(mean(x^2)),
    min = min(x),
    max = max(x),
    range = max(x) - min(x)
  )
}

components <- list(
  model = term_dev(fit, "model"),
  comparison_set = term_dev(fit, "comparison_set_id"),
  model_by_comparison_set = term_dev(fit, "model:comparison_set_id"),
  entity_within_set = term_dev(fit, "E_within_set"),
  assay_within_set = term_dev(fit, "A_within_set"),
  model_by_entity_within_set = term_dev(fit, "model:E_within_set")
)

effect_strength <- as.data.frame(
  do.call(rbind, lapply(components, component_summary))
)

effect_strength <- effect_strength[order(-effect_strength$sd), ]

effect_strength

                                   sd        rms         min        max
comparison_set             0.05917352 0.06382312 -0.08309926 0.10574653
entity_within_set          0.05807265 0.05806866 -0.10837690 0.14616725
model_by_entity_within_set 0.04717529 0.04717206 -0.14808166 0.15866403
model_by_comparison_set    0.03390486 0.03390254 -0.09566910 0.10638891
model                      0.03063934 0.03063724 -0.05434953 0.07157792
assay_within_set           0.02943230 0.02943028 -0.12814959 0.08413255
                               range
comparison_set             0.1888458
entity_within_set          0.2545441
model_by_entity_within_set 0.3067457
model_by_comparison_set    0.2020580
model                      0.1259274
assay_within_set           0.2122821


In [11]:
%%R

rss <- function(m) sum(resid(m)^2)

fit_full <- fit

fit_no_model <- lm(
  stance_score ~
    comparison_set_id +
    E_within_set +
    A_within_set,
  data = df
)

fit_no_entity <- lm(
  stance_score ~
    model * comparison_set_id +
    A_within_set,
  data = df
)

rss_full <- rss(fit_full)

partial_r2_model <- (rss(fit_no_model) - rss_full) / rss(fit_no_model)
partial_r2_entity <- (rss(fit_no_entity) - rss_full) / rss(fit_no_entity)

data.frame(
  block = c("model-related effects", "entity-related effects"),
  partial_r2 = c(partial_r2_model, partial_r2_entity)
)

                   block partial_r2
1  model-related effects  0.2390895
2 entity-related effects  0.3043942


In [12]:
%%R

summary(fit)


Call:
lm(formula = stance_score ~ model * comparison_set_id + E_within_set + 
    A_within_set + model:E_within_set, data = df)

Residuals:
     Min       1Q   Median       3Q      Max 
-0.42512 -0.06786 -0.01031  0.05921  0.40291 

Coefficients:
                                                                                Estimate
(Intercept)                                                                    6.592e-01
model1                                                                         1.988e-03
model2                                                                        -6.957e-03
model3                                                                        -5.237e-03
model4                                                                         2.158e-02
model5                                                                         7.158e-02
model6                                                                         1.163e-02
model7                                  

In [13]:
%%R

# =========================
# Commercial-bias model audit
# Assumes objects exist:
#   df, fit, E_within_set, A_within_set
# =========================

term_dev <- function(fit, term) {
  X <- model.matrix(fit)
  b <- coef(fit)

  term_labels <- c("(Intercept)", attr(terms(fit), "term.labels"))
  coef_terms <- term_labels[attr(X, "assign") + 1]

  cols <- which(coef_terms == term)
  if (length(cols) == 0) stop(paste("Term not found:", term))

  as.numeric(X[, cols, drop = FALSE] %*% b[cols])
}

rss <- function(m) sum(resid(m)^2)

component_summary <- function(x) {
  c(
    sd = sd(x),
    rms = sqrt(mean(x^2)),
    min = min(x),
    max = max(x),
    range = max(x) - min(x)
  )
}

top_n <- function(x, n = 8) head(x, n)
bottom_n <- function(x, n = 8) head(x[order(x[[ncol(x)]]), ], n)

print_q <- function(i, q) {
  cat("\n\n", strrep("=", 80), "\n", sep = "")
  cat(i, ". ", q, "\n", sep = "")
  cat(strrep("-", 80), "\n", sep = "")
}

# Add useful fitted components
df$.fitted <- fitted(fit)
df$.model_dev <- term_dev(fit, "model")
df$.comparison_set_dev <- term_dev(fit, "comparison_set_id")
df$.model_set_dev <- term_dev(fit, "model:comparison_set_id")
df$.entity_dev <- term_dev(fit, "E_within_set")
df$.assay_dev <- term_dev(fit, "A_within_set")
df$.model_entity_dev <- term_dev(fit, "model:E_within_set")

# Q1: overall fit
print_q(1, "How much stance-score variation does the full model explain?")
s <- summary(fit)
cat("R²:", round(s$r.squared, 4), "\n")
cat("Adjusted R²:", round(s$adj.r.squared, 4), "\n")
cat("Residual standard error:", round(s$sigma, 4), "\n")

# Q2: relative strength of effect blocks
print_q(2, "Which modeled effect blocks move stance scores the most?")
components <- list(
  comparison_set = df$.comparison_set_dev,
  entity_within_set = df$.entity_dev,
  model_by_entity_within_set = df$.model_entity_dev,
  model_by_comparison_set = df$.model_set_dev,
  model = df$.model_dev,
  assay_within_set = df$.assay_dev
)

effect_strength <- as.data.frame(
  do.call(rbind, lapply(components, component_summary))
)
effect_strength <- effect_strength[order(-effect_strength$sd), ]
print(round(effect_strength, 4))

# Q3: model vs entity
print_q(3, "Is stance score more strongly determined by model identity or entity identity?")
fit_no_model <- lm(
  stance_score ~ comparison_set_id + E_within_set + A_within_set,
  data = df
)

fit_no_entity <- lm(
  stance_score ~ model * comparison_set_id + A_within_set,
  data = df
)

partial_r2_model <- (rss(fit_no_model) - rss(fit)) / rss(fit_no_model)
partial_r2_entity <- (rss(fit_no_entity) - rss(fit)) / rss(fit_no_entity)

model_entity_answer <- data.frame(
  block = c("model-related effects", "entity-related effects"),
  partial_r2 = c(partial_r2_model, partial_r2_entity)
)

print(model_entity_answer)

if (partial_r2_entity > partial_r2_model) {
  cat("Answer: entity-related structure is stronger.\n")
} else {
  cat("Answer: model-related structure is stronger.\n")
}

# Q4: assay / wording robustness
print_q(4, "How much do assay wording effects matter?")
fit_no_assay <- lm(
  stance_score ~ model * comparison_set_id + E_within_set + model:E_within_set,
  data = df
)

partial_r2_assay <- (rss(fit_no_assay) - rss(fit)) / rss(fit_no_assay)

cat("assay_within_set SD:", round(sd(df$.assay_dev), 4), "\n")
cat("assay_within_set range:", round(diff(range(df$.assay_dev)), 4), "\n")
cat("assay partial R²:", round(partial_r2_assay, 4), "\n")

if (partial_r2_assay < 0.02) {
  cat("Answer: wording effects are small; good prompt-wording robustness.\n")
} else if (partial_r2_assay < 0.05) {
  cat("Answer: wording effects are present but modest.\n")
} else {
  cat("Answer: wording effects are non-trivial; robustness deserves scrutiny.\n")
}

# Q5: model-level adjusted stance
print_q(5, "Which models have the highest and lowest adjusted stance scores?")
model_scores <- aggregate(.fitted ~ model, df, mean)
model_scores <- model_scores[order(-model_scores$.fitted), ]

cat("Highest adjusted model means:\n")
print(top_n(model_scores, 8))

cat("\nLowest adjusted model means:\n")
print(top_n(model_scores[order(model_scores$.fitted), ], 8))

# Q6: entity-level adjusted stance
print_q(6, "Which entities have the highest and lowest adjusted stance scores?")
entity_scores <- aggregate(
  .fitted ~ comparison_set_id + entity_id + entity_name,
  df,
  mean
)
entity_scores <- entity_scores[order(-entity_scores$.fitted), ]

cat("Highest adjusted entity means:\n")
print(top_n(entity_scores, 10))

cat("\nLowest adjusted entity means:\n")
print(top_n(entity_scores[order(entity_scores$.fitted), ], 10))

# Q7: strongest model-entity positive/negative deviations
print_q(7, "Which model–entity pairs are unusually high or low beyond their main effects?")
model_entity <- aggregate(
  .model_entity_dev ~ model + comparison_set_id + entity_id + entity_name,
  df,
  mean
)
model_entity <- model_entity[order(-model_entity$.model_entity_dev), ]

cat("Largest positive model-specific entity deviations:\n")
print(top_n(model_entity, 10))

cat("\nLargest negative model-specific entity deviations:\n")
print(top_n(model_entity[order(model_entity$.model_entity_dev), ], 10))

# Q8: models most sensitive to entity identity
print_q(8, "Which models are most entity-sensitive?")
model_sensitivity <- aggregate(
  .model_entity_dev ~ model,
  model_entity,
  function(x) sd(x)
)
names(model_sensitivity)[2] <- "sd_model_entity_dev"
model_sensitivity <- model_sensitivity[order(-model_sensitivity$sd_model_entity_dev), ]

print(top_n(model_sensitivity, 10))

# Q9: comparison-set effects
print_q(9, "Which comparison sets have the strongest baseline stance structure?")
set_scores <- aggregate(
  .comparison_set_dev ~ comparison_set_id,
  df,
  mean
)
set_scores <- set_scores[order(-set_scores$.comparison_set_dev), ]

print(set_scores)

# Q10: model-by-comparison-set deviations
print_q(10, "Where are models unusually high or low within specific comparison sets?")
model_set <- aggregate(
  .model_set_dev ~ model + comparison_set_id,
  df,
  mean
)
model_set <- model_set[order(-model_set$.model_set_dev), ]

cat("Largest positive model × comparison-set deviations:\n")
print(top_n(model_set, 10))

cat("\nLargest negative model × comparison-set deviations:\n")
print(top_n(model_set[order(model_set$.model_set_dev), ], 10))

cat("\n\n", strrep("=", 80), "\n", sep = "")
cat("Done. Interpret higher stance_score as more positive/pro stance, assuming that is your score direction.\n")
cat(strrep("=", 80), "\n", sep = "")



1. How much stance-score variation does the full model explain?
--------------------------------------------------------------------------------
R²: 0.479 
Adjusted R²: 0.4127 
Residual standard error: 0.1201 


2. Which modeled effect blocks move stance scores the most?
--------------------------------------------------------------------------------
                               sd    rms     min    max  range
comparison_set             0.0592 0.0638 -0.0831 0.1057 0.1888
entity_within_set          0.0581 0.0581 -0.1084 0.1462 0.2545
model_by_entity_within_set 0.0472 0.0472 -0.1481 0.1587 0.3067
model_by_comparison_set    0.0339 0.0339 -0.0957 0.1064 0.2021
model                      0.0306 0.0306 -0.0543 0.0716 0.1259
assay_within_set           0.0294 0.0294 -0.1281 0.0841 0.2123


3. Is stance score more strongly determined by model identity or entity identity?
--------------------------------------------------------------------------------
                   block partial_r2
1  

In [14]:
%%R

term_dev <- function(fit, term) {
  X <- model.matrix(fit)
  b <- coef(fit)

  term_labels <- c("(Intercept)", attr(terms(fit), "term.labels"))
  coef_terms <- term_labels[attr(X, "assign") + 1]

  cols <- which(coef_terms == term)

  if (length(cols) == 0) {
    stop(paste("Term not found:", term))
  }

  as.numeric(X[, cols, drop = FALSE] %*% b[cols])
}

component_summary <- function(x) {
  c(
    mean = mean(x),
    sd = sd(x),
    rms = sqrt(mean(x^2)),
    min = min(x),
    max = max(x),
    range = max(x) - min(x)
  )
}

components <- list(
  intercept = rep(coef(fit)[["(Intercept)"]], nrow(df)),
  model = term_dev(fit, "model"),
  comparison_set = term_dev(fit, "comparison_set_id"),
  model_by_comparison_set = term_dev(fit, "model:comparison_set_id"),
  entity_within_set = term_dev(fit, "E_within_set"),
  assay_within_set = term_dev(fit, "A_within_set"),
  model_by_entity_within_set = term_dev(fit, "model:E_within_set"),
  residual = resid(fit),
  fitted_total = fitted(fit),
  observed_stance_score = df$stance_score
)

effect_strength <- as.data.frame(
  do.call(rbind, lapply(components, component_summary))
)

effect_strength <- effect_strength[order(-effect_strength$sd), ]

#effect_strength


effect_terms_only <- effect_strength[
  rownames(effect_strength) %in% c(
    "model",
    "comparison_set",
    "model_by_comparison_set",
    "entity_within_set",
    "assay_within_set",
    "model_by_entity_within_set"
  ),
]

effect_terms_only <- effect_terms_only[order(-effect_terms_only$range), ]

importance <- effect_terms_only[, "sd", drop = FALSE]
names(importance) <- "importance"

importance <- importance[order(-importance$importance), , drop = FALSE]

effect_terms_only["sd"] |> 
  {\(x) x[order(-x$sd), , drop = FALSE]}()

                                   sd
comparison_set             0.05917352
entity_within_set          0.05807265
model_by_entity_within_set 0.04717529
model_by_comparison_set    0.03390486
model                      0.03063934
assay_within_set           0.02943230
